# Propensity scores: practice notebook

**Causal Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

The code from the module's web page with `statsmodels`, the sales-rep case step by step, and the job-training data with Dehejia and Wahba's score. Exercises are marked `# YOUR CODE HERE`.

**In Colab:** *File → Save a copy in Drive* first. Run cells with **Shift+Enter**.

## 0. Setup

Two data files from the module's website: `churn.csv` (7,218 telecom customers) and `doctors.csv` (20,000 simulated doctors; specialty codes 0–5 = GP, FP, IM, Endo, OB/GYN, Card; region codes 0–3 = Northeast, Midwest, South, West). The cell looks for local copies first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/A4/"   # the module's website, e.g. "https://<user>.github.io/<repo>/A4/" (filled in when the site is published)

def load(name):
    for p in [Path(name), Path("data") / name, Path("web/data") / name]:
        if p.exists():
            return pd.read_csv(p)
    return pd.read_csv(SITE + "data/" + name)

churn = load("churn.csv")
doc = load("doctors.csv")
doc["specialty"] = doc["specialty"].map(dict(enumerate(["GP", "FP", "IM", "Endo", "OB/GYN", "Card"])))
doc["region"] = doc["region"].map(dict(enumerate(["Northeast", "Midwest", "South", "West"])))
print(churn.shape, doc.shape)
doc.head()

## Part A. Yes/no outcomes: the churn data

### A1. Linear probability model, logit, average marginal effects

In [ ]:
f = "Churn ~ MonthsToDate + MonthlyCharges + OneYear + TwoYear + SeniorCitizen + Partner + Fiber + PaperlessBilling"
lpm = smf.ols(f, data=churn).fit(cov_type="HC1")        # robust SEs for a 0/1 outcome
logit = smf.logit(f, data=churn).fit(disp=0)
ame = logit.get_margeff(at="overall")
table = pd.DataFrame({"LPM": lpm.params[1:], "logit": logit.params[1:], "logit AME": ame.margeff}).round(4)
table

In [ ]:
churn["p_lpm"], churn["p_logit"] = lpm.predict(churn), logit.predict(churn)
print(f"LPM predictions below 0: {(churn.p_lpm < 0).mean():.1%}")
print(f"logit predictions from {churn.p_logit.min():.3f} to {churn.p_logit.max():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
ax[0].hist(churn.p_lpm, bins=50, color="indianred"); ax[0].axvline(0, color="k"); ax[0].set_title("Linear probability model")
ax[1].hist(churn.p_logit, bins=50, color="steelblue"); ax[1].set_title("Logit")
plt.show()

### A2. A dummy: predict twice and subtract

A "typical" customer: median months and charges, the most common value of every other variable, on a month-to-month contract.

In [ ]:
typical = churn.iloc[[0]].copy()
for c in ["MonthsToDate", "MonthlyCharges"]:
    typical[c] = churn[c].median()
for c in ["SeniorCitizen", "Partner", "Fiber", "PaperlessBilling"]:
    typical[c] = churn[c].mode()[0]
typical["OneYear"], typical["TwoYear"] = 0, 0
p0 = float(logit.predict(typical).iloc[0])
typical["TwoYear"] = 1
p1 = float(logit.predict(typical).iloc[0])
print(f"month-to-month {p0:.3f} -> two-year {p1:.3f}: {100 * (p1 - p0):+.1f} points (LPM: {100 * lpm.params['TwoYear']:+.1f})")

### A3. The same logit from scratch (the page's Python cell)

In [ ]:
y = churn["Churn"].to_numpy()
names = ["MonthsToDate", "MonthlyCharges", "OneYear", "TwoYear", "SeniorCitizen", "Partner", "Fiber", "PaperlessBilling"]
X = np.column_stack([np.ones(len(y))] + [churn[v].to_numpy() for v in names])

def newton_logit(X, y, steps=25):                     # maximum likelihood by Newton's method
    b = np.zeros(X.shape[1])
    for _ in range(steps):
        p = 1 / (1 + np.exp(-X @ b))
        b += np.linalg.solve(X.T @ (X * (p * (1 - p))[:, None]), X.T @ (y - p))
    return b

b = newton_logit(X, y)
print(np.round(b, 3))
print(np.round(logit.params.to_numpy(), 3))            # the same numbers

**Exercise A.** (1) Make a table of the share of customers flagged, the share of churners caught, and the share of stayers flagged for thresholds 0.2, 0.3, 0.4, 0.5. (2) Fit the same model by probit (`smf.probit`). How do its coefficients compare with the logit's, and its average marginal effects?

In [ ]:
# YOUR CODE HERE

## Part B. The sales-rep case, step by step

### B1. The naive comparison, and the propensity score

The score is built from the treatment and the covariates only: the outcome (`scripts_t2`) stays in escrow until the design is done.

In [ ]:
print(doc.groupby("visited")[["prior_rx", "scripts_t2"]].mean().round(1))
print(f"naive difference: {doc.scripts_t2[doc.visited == 1].mean() - doc.scripts_t2[doc.visited == 0].mean():.2f}")

X_RHS = "prior_rx + C(specialty) + C(region) + years + group_practice + hmo_share"
ps = smf.logit("visited ~ " + X_RHS, data=doc).fit(disp=0)
doc["e"] = ps.predict(doc)
doc["lin"] = np.log(doc.e / (1 - doc.e))
print(f"coefficient on prior prescriptions: {ps.params['prior_rx']:.4f}")

### B2. Overlap and common support

In [ ]:
v, u = doc[doc.visited == 1], doc[doc.visited == 0]
lo, hi = max(v.e.min(), u.e.min()), min(v.e.max(), u.e.max())
print(f"common support [{lo:.3f}, {hi:.3f}]: drops {(v.e > hi).sum()} visited and {(u.e < lo).sum()} unvisited doctors")
fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
ax[0].hist(u.e, bins=50, color="steelblue"); ax[0].set_title("not visited", loc="left")
ax[1].hist(v.e, bins=50, color="darkorange"); ax[1].set_title("visited", loc="left")
ax[1].set_xlabel("estimated propensity score")
plt.show()
cs = doc[(doc.e >= lo) & (doc.e <= hi)].copy()

### B3. Balance inside one bin of the score

In [ ]:
b = doc[(doc.e >= 0.5) & (doc.e < 0.6)]
print(b.groupby("visited")[["prior_rx", "years", "hmo_share"]].mean().round(2))
print(pd.crosstab(b.specialty, b.visited, normalize="columns").round(3))

### B4. Subclassification: five strata

In [ ]:
def strata_att(data, K):
    data = data.assign(s=pd.qcut(data.e, K, labels=False))
    g = data.groupby(["s", "visited"]).scripts_t2.mean().unstack()
    n1 = data[data.visited == 1].groupby("s").size()
    n = data.groupby("s").size()
    diff = g[1] - g[0]
    return np.average(diff, weights=n1), np.average(diff, weights=n)

att5, ate5 = strata_att(cs, 5)
print(f"five strata: ATT {att5:.2f}, ATE {ate5:.2f}   (truth: ATT 1.55, ATE 3.05 on the common support)")

### B5. Matching on the log-odds of the score, with a caliper

In [ ]:
v, u = doc[doc.visited == 1], doc[doc.visited == 0]
caliper = 0.2 * doc.lin.std()
nn = NearestNeighbors(n_neighbors=1).fit(u[["lin"]])
dist, idx = nn.kneighbors(v[["lin"]])
ok = dist[:, 0] <= caliper
matched = u.iloc[idx[ok, 0]]
att_match = (v.scripts_t2.to_numpy()[ok] - matched.scripts_t2.to_numpy()).mean()
print(f"nearest neighbor within {caliper:.3f}: ATT {att_match:.2f}; {(~ok).sum()} visited doctors without a match; "
      f"{matched.index.nunique():,} distinct unvisited doctors used")

### B6. Did it work? Standardized mean differences (a Love plot)

In [ ]:
covs = pd.get_dummies(doc[["prior_rx", "years", "group_practice", "hmo_share", "specialty", "region"]], dtype=float)

def smd(a, b, w=None):
    s = np.sqrt((a.var() + b.var()) / 2)
    return (a.mean() - np.average(b, weights=w)) / s

vc, uc = covs[doc.visited == 1], covs[doc.visited == 0]
w_att = (doc.e / (1 - doc.e))[doc.visited == 0]
bal = pd.DataFrame({"before": [smd(vc[c], uc[c]) for c in covs],
                    "after matching": [smd(vc[c][ok], covs.loc[matched.index, c]) for c in covs],
                    "after weighting": [smd(vc[c], uc[c], w_att) for c in covs]}, index=covs.columns)
print(bal.round(3))
fig, ax = plt.subplots(figsize=(6, 5))
for col, m in zip(bal.columns, ["o", "s", "^"]):
    ax.scatter(bal[col].abs(), range(len(bal)), marker=m, label=col)
ax.axvline(0.1, ls="--", color="grey"); ax.set_yticks(range(len(bal))); ax.set_yticklabels(bal.index, fontsize=8)
ax.set_xlabel("|standardized mean difference|"); ax.legend()
plt.show()

### B7. Weighting, trimming, and the doubly robust estimate

In [ ]:
def ipw(data):
    e, D, Y = data.e.to_numpy(), data.visited.to_numpy(), data.scripts_t2.to_numpy()
    ate = np.average(Y[D == 1], weights=1 / e[D == 1]) - np.average(Y[D == 0], weights=1 / (1 - e[D == 0]))
    att = Y[D == 1].mean() - np.average(Y[D == 0], weights=(e / (1 - e))[D == 0])
    return ate, att, np.where(D == 1, 1 / e, 1 / (1 - e)).max()

ate, att, wmax = ipw(doc)
print(f"weighting: ATE {ate:.2f}, ATT {att:.2f}, largest weight {wmax:.0f}")
trim = doc[(doc.e >= 0.05) & (doc.e <= 0.95)]
print("trimmed to [0.05, 0.95]: ATE {:.2f}, ATT {:.2f}, largest weight {:.0f}".format(*ipw(trim)))

mu1 = smf.ols("scripts_t2 ~ " + X_RHS, data=doc[doc.visited == 1]).fit().predict(doc)
mu0 = smf.ols("scripts_t2 ~ " + X_RHS, data=doc[doc.visited == 0]).fit().predict(doc)
D, Y, e = doc.visited, doc.scripts_t2, doc.e
aipw = np.mean(mu1 + D * (Y - mu1) / e - (mu0 + (1 - D) * (Y - mu0) / (1 - e)))
ols = smf.ols("scripts_t2 ~ visited + " + X_RHS, data=doc).fit().params["visited"]
print(f"OLS with controls {ols:.2f}; doubly robust ATE {aipw:.2f}   (truth: ATE 3.05, ATT 1.53)")

### B8. All of it in plain NumPy (the page's Python cell)

In [ ]:
d = doc
Dn, Yn = d.visited.to_numpy(), d.scripts_t2.to_numpy()
Xn = np.column_stack([np.ones(len(d)), d.prior_rx, pd.get_dummies(d.specialty, dtype=float).drop(columns="GP"),
                      pd.get_dummies(d.region, dtype=float).drop(columns="Northeast"), d.years, d.group_practice, d.hmo_share])
en = 1 / (1 + np.exp(-Xn @ newton_logit(Xn, Dn)))
t, c = Dn == 1, Dn == 0
w = en / (1 - en)
print(f"naive {Yn[t].mean() - Yn[c].mean():.2f}; weighting (ATT) {Yn[t].mean() - np.average(Yn[c], weights=w[c]):.2f}")

**Exercise B.** (1) Compute the five-, ten-, and twenty-strata ATT with `strata_att` and compare with the truth (1.55). (2) Re-do the matching with ten neighbors inside the caliper. (3) Trim at [0.1, 0.9]: how many doctors are dropped, and what happens to the ATE estimate? Why does the *true* effect also change?

In [ ]:
# YOUR CODE HERE

## Part C. The job-training data with the score

185 trainees from the randomized NSW program and 15,992 men from the CPS. The experiment's answer: about \$1,794. Files from Rajeev Dehejia's page at NBER.

In [ ]:
cols = ["treat", "age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75", "re78"]
URL = "https://users.nber.org/~rdehejia/data/"

def read(name):
    for p in [Path(name), Path("data") / name, Path("../../../L07_matching/data") / name]:
        if p.exists():
            return pd.read_csv(p, sep=r"\s+", header=None, names=cols)
    return pd.read_csv(URL + name, sep=r"\s+", header=None, names=cols)

nsw_t, nsw_c, cps = read("nswre74_treated.txt"), read("nswre74_control.txt"), read("cps_controls.txt")
print(f"experiment: {nsw_t.re78.mean() - nsw_c.re78.mean():,.0f}")
d = pd.concat([nsw_t, cps], ignore_index=True)
d["u74"], d["u75"] = (d.re74 == 0).astype(int), (d.re75 == 0).astype(int)
DW = "treat ~ age + I(age**2) + I(age**3) + educ + I(educ**2) + married + nodegree + black + hisp + re74 + re75 + u74 + u75 + I(educ*re74)"
d["e"] = smf.logit(DW, data=d).fit(disp=0, maxiter=300).predict(d)
t, c = d[d.treat == 1], d[d.treat == 0]
print(f"raw difference {t.re78.mean() - c.re78.mean():,.0f}; CPS men with a score below 0.01: {(c.e < 0.01).mean():.0%}")

In [ ]:
w = c.e / (1 - c.e)
print(f"weighting (ATT): {t.re78.mean() - np.average(c.re78, weights=w):,.0f}")
nn = NearestNeighbors(n_neighbors=1).fit(np.log(c.e / (1 - c.e)).to_frame())
_, i1 = nn.kneighbors(np.log(t.e / (1 - t.e)).to_frame())
m = c.iloc[i1[:, 0]]
print(f"one-to-one matching: {(t.re78.to_numpy() - m.re78.to_numpy()).mean():,.0f} using {m.index.nunique()} distinct CPS men")
for x in ["age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75"]:
    s = np.sqrt((t[x].var() + m[x].var()) / 2)
    print(f"  {x:9s} SMD after matching {(t[x].mean() - m[x].mean()) / s:+.2f}")

**Exercise C.** A placebo test: re-estimate the score *without* 1975 earnings (drop `re75` and `u75`), then use 1975 earnings as the outcome. The training started after 1975, so the true "effect" is zero. Compare the raw difference with the weighted (ATT) difference.

In [ ]:
# YOUR CODE HERE